# 🎯 COMPLETE FSL MODEL TRAINING PIPELINE
## All-in-One: Dataset → Preprocess → Train → Test → Save

This notebook handles the entire training pipeline:
1. ✅ Upload/Setup your video dataset
2. ✅ Preprocess videos to 30 frames
3. ✅ Extract MediaPipe landmarks
4. ✅ Train LSTM model
5. ✅ Test model accuracy
6. ✅ Save model for deployment

**Just run all cells in order!**

---
## 📦 STEP 1: Install Dependencies

In [ ]:
!pip install mediapipe==0.9.1.0 opencv-python tensorflow scikit-learn matplotlib numpy

import cv2
import numpy as np
import os
import mediapipe as mp
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv1D
from tensorflow.keras.callbacks import TensorBoard
import time

print("✅ All dependencies installed!")

---
## 🎬 STEP 2: Upload Your Dataset

**Option A:** Upload from local computer  
**Option B:** Mount Google Drive (recommended for large datasets)

In [ ]:
# Option A: Mount Google Drive (RECOMMENDED)
from google.colab import drive
drive.mount('/content/drive')

# After mounting, copy your videos from Drive
# Uncomment and modify the path below:
# !cp -r /content/drive/MyDrive/my_sign_videos /content/videos

print("✅ Google Drive mounted!")
print("📁 Now copy your videos to /content/videos/")

In [ ]:
# Option B: Upload directly from computer (for small datasets)
# Uncomment to use:
# from google.colab import files
# import zipfile

# print("Upload your videos.zip file:")
# uploaded = files.upload()

# # Extract the zip
# for filename in uploaded.keys():
#     with zipfile.ZipFile(filename, 'r') as zip_ref:
#         zip_ref.extractall('/content/videos')

# print("✅ Videos uploaded and extracted!")

In [ ]:
# Verify your video structure
!ls -R /content/videos/

print("\n📋 Expected structure:")
print("/content/videos/")
print("  ├── sign1/")
print("  │   ├── 0.mp4")
print("  │   ├── 1.mp4")
print("  │   └── ...")
print("  ├── sign2/")
print("  └── ...")

---
## ⚙️ STEP 3: Configure Your Dataset

**🚨 IMPORTANT: Update these variables to match YOUR dataset!**

In [ ]:
# ========================================
# 🔧 CONFIGURE THESE FOR YOUR DATASET
# ========================================

# List all your sign language gestures
actions = np.array(['hello', 'thank_you', 'please', 'sorry', 'help'])  # ← CHANGE THIS

# Number of videos you have per sign
no_sequences = 20  # ← CHANGE THIS (e.g., if you have 20 videos per sign)

# Frames per video (always 30)
sequence_length = 30

# Model name for saving
model_name = 'my_fsl_model.h5'  # ← CHANGE THIS if you want

# Training parameters
epochs = 2000  # Number of training epochs (adjust based on your needs)
test_split = 0.1  # 10% of data for testing

# ========================================

print(f"✅ Configuration:")
print(f"   Signs: {actions}")
print(f"   Videos per sign: {no_sequences}")
print(f"   Total videos: {len(actions) * no_sequences}")
print(f"   Frames per video: {sequence_length}")
print(f"   Model name: {model_name}")

---
## 🎥 STEP 4: Preprocess Videos to 30 Frames

This normalizes all videos to exactly 30 frames.

In [ ]:
def count_frames(video_path):
    """Count the number of frames in a video."""
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frame_count

def preprocess_video_to_30_frames(input_path, output_path, target_frames=30):
    """Squeeze or stretch video to exactly 30 frames."""
    cap = cv2.VideoCapture(input_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Calculate which frames to extract
    if total_frames > target_frames:
        # Squeeze: sample frames evenly
        frame_indices = np.linspace(0, total_frames - 1, target_frames, dtype=int)
    else:
        # Stretch: duplicate frames
        frame_indices = np.round(np.linspace(0, total_frames - 1, target_frames)).astype(int)
    
    # Write output video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    
    cap.release()
    
    # Write selected frames
    for idx in frame_indices:
        out.write(frames[idx])
    
    out.release()
    return True

# Create output directory
os.makedirs('videos_30frames', exist_ok=True)

print("🎬 Preprocessing videos to 30 frames...\n")

for action in actions:
    os.makedirs(f'videos_30frames/{action}', exist_ok=True)
    
    for sequence in range(no_sequences):
        input_path = f'videos/{action}/{sequence}.mp4'
        output_path = f'videos_30frames/{action}/{sequence}.mp4'
        
        if os.path.exists(input_path):
            original_frames = count_frames(input_path)
            preprocess_video_to_30_frames(input_path, output_path)
            print(f"✅ {action}/{sequence}.mp4: {original_frames} → 30 frames")
        else:
            print(f"⚠️  Missing: {input_path}")

print("\n✅ All videos preprocessed to 30 frames!")

---
## 🤖 STEP 5: Extract MediaPipe Landmarks

This extracts pose, hand, and face landmarks from each frame.

In [ ]:
# MediaPipe setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    """Process image with MediaPipe."""
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def extract_keypoints(results):
    """Extract landmarks as numpy array (258 values)."""
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

# Create MP_Data directory
DATA_PATH = 'MP_Data'
os.makedirs(DATA_PATH, exist_ok=True)

for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

print("🤖 Extracting MediaPipe landmarks...\n")

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for action in actions:
        for sequence in range(no_sequences):
            video_path = f'videos_30frames/{action}/{sequence}.mp4'
            
            if not os.path.exists(video_path):
                print(f"⚠️  Missing: {video_path}")
                continue
            
            cap = cv2.VideoCapture(video_path)
            frame_num = 0
            
            while cap.isOpened() and frame_num < sequence_length:
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Process with MediaPipe
                image, results = mediapipe_detection(frame, holistic)
                
                # Extract and save keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)
                
                frame_num += 1
            
            cap.release()
            print(f"✅ {action}/{sequence}: Extracted {frame_num} frames")

print("\n✅ All landmarks extracted and saved to MP_Data/")

---
## 📊 STEP 6: Prepare Training Data

In [ ]:
# Create label map
label_map = {label: num for num, label in enumerate(actions)}
print("🏷️  Label Map:")
print(label_map)

# Load sequences and labels
sequences, labels = [], []

print("\n📦 Loading landmark data...")

for action in actions:
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), f"{frame_num}.npy"))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

X = np.array(sequences)
y = to_categorical(labels).astype(int)

print(f"\n✅ Data loaded:")
print(f"   X shape: {X.shape}  (samples, frames, features)")
print(f"   y shape: {y.shape}  (samples, classes)")

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_split, random_state=42)

print(f"\n📊 Train/Test Split:")
print(f"   Training samples: {len(X_train)}")
print(f"   Testing samples: {len(X_test)}")

---
## 🧠 STEP 7: Build and Train Model

In [ ]:
# Setup TensorBoard
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

# Build model
print("🏗️  Building model architecture...\n")

model = Sequential()
model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(30, 258)))
model.add(LSTM(64, return_sequences=True, activation='relu'))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

model.summary()

print(f"\n🚀 Starting training for {epochs} epochs...")
print("⏳ This may take a while. Go grab a coffee! ☕\n")

# Train model
history = model.fit(X_train, y_train, epochs=epochs, callbacks=[tb_callback], validation_split=0.1, verbose=1)

print("\n✅ Training complete!")

---
## 📈 STEP 8: Evaluate Model

In [ ]:
# Make predictions
print("🔍 Evaluating model on test set...\n")

yhat = model.predict(X_test)
ytrue = np.argmax(y_test, axis=1).tolist()
yhat_classes = np.argmax(yhat, axis=1).tolist()

# Calculate accuracy
accuracy = accuracy_score(ytrue, yhat_classes)

print(f"🎯 Test Accuracy: {accuracy * 100:.2f}%\n")

# Confusion matrix
print("📊 Confusion Matrix (per class):")
cm = multilabel_confusion_matrix(ytrue, yhat_classes)
for i, action in enumerate(actions):
    print(f"\n{action}:")
    print(cm[i])

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['categorical_accuracy'], label='Training Accuracy')
plt.plot(history.history['val_categorical_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

print("\n✅ Evaluation complete!")

---
## 💾 STEP 9: Save Model

In [ ]:
# Save model
model.save(model_name)
print(f"✅ Model saved as: {model_name}")

# Save action labels
np.save('action_labels.npy', actions)
print(f"✅ Action labels saved as: action_labels.npy")

# Display file info
model_size = os.path.getsize(model_name) / (1024 * 1024)  # MB
print(f"\n📦 Model size: {model_size:.2f} MB")
print(f"📋 Actions: {list(actions)}")
print(f"🎯 Test Accuracy: {accuracy * 100:.2f}%")

---
## 🧪 STEP 10: Test Model (Optional)

Quick test to verify the model works.

In [ ]:
# Test on a random sample
print("🧪 Testing model on random samples...\n")

for i in range(min(5, len(X_test))):
    sample = X_test[i:i+1]
    prediction = model.predict(sample)
    predicted_class = np.argmax(prediction)
    true_class = np.argmax(y_test[i])
    confidence = prediction[0][predicted_class] * 100
    
    print(f"Sample {i+1}:")
    print(f"  True: {actions[true_class]}")
    print(f"  Predicted: {actions[predicted_class]} ({confidence:.1f}% confidence)")
    print(f"  {'✅ Correct' if predicted_class == true_class else '❌ Wrong'}\n")

---
## 📥 STEP 11: Download Model

Download your trained model to use in the webapp!

In [ ]:
from google.colab import files

# Download model
print("📥 Downloading model...")
files.download(model_name)

# Download action labels
print("📥 Downloading action labels...")
files.download('action_labels.npy')

print("\n✅ Downloads complete!")
print("\n🎉 SUCCESS! Your model is ready to use!")
print("\n📝 Next steps:")
print("   1. Replace the model file in your webapp's 'models/' folder")
print("   2. Update the action labels in your test scripts")
print("   3. Test with test_live.py or test_video.py")
print("   4. Deploy your custom sign language translator!")

---
## 🎉 CONGRATULATIONS!

You've successfully trained your custom Filipino Sign Language model!

### 📊 Summary:
- ✅ Preprocessed videos to 30 frames
- ✅ Extracted MediaPipe landmarks
- ✅ Trained LSTM model
- ✅ Evaluated model accuracy
- ✅ Saved model for deployment

### 🚀 What's Next?
1. **Download** your model files (already done above)
2. **Replace** the old model in your webapp
3. **Test** with real-time video or recorded videos
4. **Deploy** your custom translator!

### 💡 Tips:
- If accuracy is low, try collecting more videos per sign (30-50 recommended)
- Ensure good lighting and clear hand gestures in your videos
- Train for more epochs if the model hasn't converged
- Add data augmentation for better generalization

---

**Need help?** Check the troubleshooting section in COLAB_WORKFLOW_GUIDE.md